# Property prediction with ViralForsight

## Extract RBD sequences from generated S1 sequences

In [ ]:
generated_s1s_dir = 'Path to the folder containing the generated S1 sequences per accession'
rbd_start = 357  # Start index of the RBD in the aligned S1 sequence (residue 331)
rbd_end = 558    # End index of the RBD in the aligned S1 sequence (residue 531 + 1)

In [ ]:
import os
from tqdm import tqdm

sequences = {}
for subdir in os.listdir(generated_s1s_dir):
    raw_seqs_dir = os.path.join(generated_s1s_dir, subdir, f'{subdir}_raw_seqs')
    if not os.path.isdir(raw_seqs_dir):
        continue
    sequences[subdir] = []
    for file in os.listdir(raw_seqs_dir):
        with open(os.path.join(raw_seqs_dir, file), 'r') as f:
            for line in f:
                sequences[subdir].append(line.strip())

rbds = {
    accession: list(set(seq[rbd_start:rbd_end] for seq in seq_list))
    for accession, seq_list in sequences.items()
}

for accession in tqdm(rbds.keys()):
    with open(os.path.join(generated_s1s_dir, f'{accession}.txt'), 'w') as f:
        for rbd in rbds[accession]:
            f.write(f'{rbd}\n')

    with open(os.path.join(generated_s1s_dir, f'{accession}.fasta'), 'w') as f:
        for seq in sequences[accession]:
            f.write(f'>{accession}\n{seq}\n')

## Download pretrained checkpoints and normalization statistics from Figshare

In [ ]:
path_to_property_model_dir = 'Path where to store the downloaded checkpoints and normalization statistics'
tmp_download_dir = 'Path to a temporary directory for downloading the Figshare zip archives'

# Figshare direct download URLs from the E2VD repository
plm_zip_url = 'https://ndownloader.figshare.com/files/46676821/checkpoint_customize.zip'             # checkpoint_customize.zip (DOI: 10.6084/m9.figshare.25932394)
property_models_zip_url = 'https://ndownloader.figshare.com/files/46617082/PerceiveEvolutionaryTrends.zip' # PerceiveEvolutionaryTrends.zip (DOI: 10.6084/m9.figshare.25913404.v1)

In [ ]:
import os
import shutil
import urllib.request
import zipfile

os.makedirs(tmp_download_dir, exist_ok=True)

# 1. Download and extract the custom BERT protein language model checkpoint (checkpoint_customize.ckpt)
plm_zip_path = os.path.join(tmp_download_dir, 'checkpoint_customize.zip')
path_to_property_model_ckpt = os.path.join(path_to_property_model_dir, 'checkpoint_customize.ckpt')
print('Downloading checkpoint_customize.zip...')
urllib.request.urlretrieve(plm_zip_url, plm_zip_path)

with zipfile.ZipFile(plm_zip_path, 'r') as zf:
    for member in zf.namelist():
        if member.endswith('checkpoint_customize.ckpt'):
            os.makedirs(os.path.dirname(path_to_property_model_ckpt), exist_ok=True)
            with zf.open(member) as src, open(path_to_property_model_ckpt, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            print(f'Extracted {path_to_property_model_ckpt}')

os.remove(plm_zip_path)

# 2. Download and extract the pretrained expression/binding models and normalization statistics
files_to_extract = {
    'PerceiveEvolutionaryTrends/trained_ckpt/expr.pth.tar': os.path.join(path_to_property_model_dir, 'expr.pth.tar'),
    'PerceiveEvolutionaryTrends/trained_ckpt/bind.pth.tar': os.path.join(path_to_property_model_dir, 'bind.pth.tar'),
    'PerceiveEvolutionaryTrends/trained_ckpt/expr_mean.npy': os.path.join(path_to_property_model_dir, 'expr/embedding_flatten_mean.npy'),
    'PerceiveEvolutionaryTrends/trained_ckpt/expr_std.npy': os.path.join(path_to_property_model_dir, 'expr/embedding_flatten_std.npy'),
    'PerceiveEvolutionaryTrends/trained_ckpt/bind_mean.npy': os.path.join(path_to_property_model_dir, 'bind/embedding_flatten_mean.npy'),
    'PerceiveEvolutionaryTrends/trained_ckpt/bind_std.npy': os.path.join(path_to_property_model_dir, 'bind/embedding_flatten_std.npy'),
}

prop_zip_path = os.path.join(tmp_download_dir, 'PerceiveEvolutionaryTrends.zip')
print('Downloading PerceiveEvolutionaryTrends.zip...')
urllib.request.urlretrieve(property_models_zip_url, prop_zip_path)

with zipfile.ZipFile(prop_zip_path, 'r') as zf:
    for zip_member, target_path in files_to_extract.items():
        os.makedirs(os.path.dirname(target_path), exist_ok=True)
        with zf.open(zip_member) as src, open(target_path, 'wb') as dst:
            shutil.copyfileobj(src, dst)
        print(f'Extracted {target_path}')

# Clean up the downloaded zip file
os.remove(prop_zip_path)
print('Download and cleanup complete!')

## Convert MindSpore BERT checkpoint to PyTorch

In [ ]:
ms_ckpt_path = 'Path to the input MindSpore checkpoint (checkpoint_customize.ckpt)'
pytorch_ckpt_path = 'Path to save the converted PyTorch checkpoint (checkpoint_customize_pytorch.pt)'

In [ ]:
import numpy as np
import torch

def read_varint(f):
    shift = 0
    result = 0
    while True:
        b = f.read(1)
        if not b:
            return None
        b = b[0]
        result |= (b & 0x7F) << shift
        if not (b & 0x80):
            return result
        shift += 7

def parse_protobuf_fields(buf):
    idx = 0
    n = len(buf)
    fields = []
    while idx < n:
        shift = 0
        tag = 0
        while True:
            b = buf[idx]
            idx += 1
            tag |= (b & 0x7F) << shift
            if not (b & 0x80):
                break
            shift += 7
        field_num = tag >> 3
        wire_type = tag & 7
        if wire_type == 0:  # varint
            val = 0
            shift = 0
            while True:
                b = buf[idx]
                idx += 1
                val |= (b & 0x7F) << shift
                if not (b & 0x80):
                    break
                shift += 7
            fields.append((field_num, val))
        elif wire_type == 2:  # length-delimited bytes
            length = 0
            shift = 0
            while True:
                b = buf[idx]
                idx += 1
                length |= (b & 0x7F) << shift
                if not (b & 0x80):
                    break
                shift += 7
            fields.append((field_num, buf[idx : idx + length]))
            idx += length
    return fields

weights = {}
with open(ms_ckpt_path, 'rb') as f:
    while True:
        tag = read_varint(f)
        if tag is None:
            break
        length = read_varint(f)
        data = f.read(length)
        fields = dict(parse_protobuf_fields(data))
        name = fields[1].decode()

        # Stop once we reach the LAMB optimizer state variables
        if name == 'scale_sense' or name.startswith('lamb_'):
            break

        tensor_fields = parse_protobuf_fields(fields[2])
        dims = [v for k, v in tensor_fields if k == 1]
        raw_bytes = b''.join(v for k, v in tensor_fields if k == 3)
        arr = np.frombuffer(raw_bytes, dtype=np.float32).copy()
        if dims:
            arr = arr.reshape(dims)
        weights[name] = torch.from_numpy(arr)

torch.save(weights, pytorch_ckpt_path)
print(f'Saved {len(weights)} parameter tensors to {pytorch_ckpt_path}')

## Predict RBD properties (ACE2 binding and expression) on CUDA

In [ ]:
input_rbd_dir = 'Path to the directory containing the extracted {accession}.txt RBD files'
output_pred_dir = 'Path to the directory where pred_{accession}.csv files will be saved'
bert_ckpt_path = 'Path to checkpoint_customize_pytorch.pt'
expr_ckpt_path = 'Path to expr.pth.tar'
bind_ckpt_path = 'Path to bind.pth.tar'
stats_dir = 'Path to data/property_prediction directory containing expr/ and bind/ normalization stats'

batch_size = 256

In [ ]:
import glob
import math
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

## Amino acid vocabulary matching E2VD's extract_embedding_local.py
MAP_DICT = {
    '[PAD]': 0, '[UNK]': 1, '[CLS]': 2, '[SEP]': 3, '[MASK]': 4,
    'L': 5, 'A': 6, 'G': 7, 'V': 8, 'E': 9, 'S': 10, 'I': 11,
    'K': 12, 'R': 13, 'D': 14, 'T': 15, 'P': 16, 'N': 17, 'Q': 18,
    'F': 19, 'Y': 20, 'M': 21, 'H': 22, 'C': 23, 'W': 24, 'X': 25
}

## Model definitions

class CustomizeBertEncoderLayer(nn.Module):
    def __init__(self, hidden_size=1024, num_heads=16, intermediate_size=4096):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.query = nn.Linear(hidden_size, hidden_size)
        self.key = nn.Linear(hidden_size, hidden_size)
        self.value = nn.Linear(hidden_size, hidden_size)
        self.attn_out_dense = nn.Linear(hidden_size, hidden_size)
        self.attn_ln = nn.LayerNorm(hidden_size, eps=1e-7)
        self.inter_dense = nn.Linear(hidden_size, intermediate_size)
        self.out_dense = nn.Linear(intermediate_size, hidden_size)
        self.out_ln = nn.LayerNorm(hidden_size, eps=1e-7)

    def forward(self, x, attn_mask=None):
        B, L, H = x.shape
        q = self.query(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.key(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.value(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.head_dim)
        if attn_mask is not None:
            scores = scores + (1.0 - attn_mask[:, None, None, :]) * (-10000.0)
        probs = torch.softmax(scores, dim=-1)
        ctx = torch.matmul(probs, v).transpose(1, 2).contiguous().view(B, L, H)
        attn_out = self.attn_ln(x + self.attn_out_dense(ctx))
        inter = F.gelu(self.inter_dense(attn_out), approximate='tanh')
        out = self.out_ln(attn_out + self.out_dense(inter))
        return out

class CustomizeBertModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.word_emb = nn.Embedding(26, 1024)
        self.pos_emb = nn.Embedding(40000, 1024)
        self.emb_ln = nn.LayerNorm(1024, eps=1e-7)
        self.layers = nn.ModuleList([CustomizeBertEncoderLayer() for _ in range(24)])

    def load_from_ms_dict(self, w):
        with torch.no_grad():
            self.word_emb.weight.copy_(w['bert.bert.bert_embedding_lookup.embedding_table'])
            self.pos_emb.weight.copy_(w['bert.bert.bert_embedding_postprocessor.full_position_embedding.embedding_table'])
            self.emb_ln.weight.copy_(w['bert.bert.bert_embedding_postprocessor.layernorm.gamma'])
            self.emb_ln.bias.copy_(w['bert.bert.bert_embedding_postprocessor.layernorm.beta'])
            for i, layer in enumerate(self.layers):
                p = f'bert.bert.bert_encoder.layers.{i}.'
                layer.query.weight.copy_(w[p + 'attention.attention.query_layer.weight'])
                layer.query.bias.copy_(w[p + 'attention.attention.query_layer.bias'])
                layer.key.weight.copy_(w[p + 'attention.attention.key_layer.weight'])
                layer.key.bias.copy_(w[p + 'attention.attention.key_layer.bias'])
                layer.value.weight.copy_(w[p + 'attention.attention.value_layer.weight'])
                layer.value.bias.copy_(w[p + 'attention.attention.value_layer.bias'])
                layer.attn_out_dense.weight.copy_(w[p + 'attention.output.dense.weight'])
                layer.attn_out_dense.bias.copy_(w[p + 'attention.output.dense.bias'])
                layer.attn_ln.weight.copy_(w[p + 'attention.output.layernorm.gamma'])
                layer.attn_ln.bias.copy_(w[p + 'attention.output.layernorm.beta'])
                layer.inter_dense.weight.copy_(w[p + 'intermediate.weight'])
                layer.inter_dense.bias.copy_(w[p + 'intermediate.bias'])
                layer.out_dense.weight.copy_(w[p + 'output.dense.weight'])
                layer.out_dense.bias.copy_(w[p + 'output.dense.bias'])
                layer.out_ln.weight.copy_(w[p + 'output.layernorm.gamma'])
                layer.out_ln.bias.copy_(w[p + 'output.layernorm.beta'])

    def forward(self, input_ids, input_mask=None):
        B, L = input_ids.shape
        pos_ids = torch.arange(L, device=input_ids.device).unsqueeze(0)
        x = self.emb_ln(self.word_emb(input_ids) + self.pos_emb(pos_ids))
        for layer in self.layers:
            x = layer(x, input_mask)
        return x

class ContextPooling(nn.Module):
    def __init__(self, seq_len, in_dim=1024):
        super().__init__()
        self.seq_len = seq_len
        self.conv = nn.Sequential(
            nn.Conv1d(in_dim, in_dim * 2, 3, stride=1, padding=1),
            nn.LayerNorm((in_dim * 2, seq_len)),
            nn.LeakyReLU(True),
            nn.Conv1d(in_dim * 2, in_dim * 2, 3, stride=1, padding=1),
            nn.LayerNorm((in_dim * 2, seq_len)),
            nn.LeakyReLU(True),
            nn.Conv1d(in_dim * 2, 2, 3, stride=1, padding=1),
            nn.LayerNorm((2, seq_len)),
            nn.LeakyReLU(True),
        )

    def forward(self, feats):
        feats_ = self.conv(feats.permute(0, 2, 1))
        s = torch.softmax(feats_[:, 0, :], dim=1)
        w = torch.softmax(feats_[:, 1, :], dim=1)
        PI = 3.1415926
        std_ = (0.1 * self.seq_len * s).unsqueeze(2)
        place = torch.arange(self.seq_len, device=feats.device, dtype=feats.dtype)
        diff_sq = (place.unsqueeze(0) - place.unsqueeze(1)).pow(2).unsqueeze(0)
        ret = pow(2 * PI, -0.5) * torch.pow(std_, -1) * torch.exp(
            -diff_sq / (1e-5 + 2 * torch.pow(std_, 2))
        )
        ret = ret / torch.max(ret, dim=2, keepdim=True)[0]
        w_all = ret * w.unsqueeze(1)
        return torch.bmm(w_all, feats)

class SoluModel(nn.Module):
    def __init__(self, seq_len=201, in_dim=1024, sa_out=1024, conv_out=1024):
        super().__init__()
        self.contextpooling = ContextPooling(seq_len, in_dim)
        self.conv = nn.Sequential(
            nn.Conv1d(in_dim, in_dim * 2, 3, stride=1, padding=1),
            nn.LayerNorm((in_dim * 2, seq_len)),
            nn.LeakyReLU(True),
            nn.Conv1d(in_dim * 2, in_dim * 2, 3, stride=1, padding=1),
            nn.LayerNorm((in_dim * 2, seq_len)),
            nn.LeakyReLU(True),
            nn.Conv1d(in_dim * 2, conv_out, 3, stride=1, padding=1),
            nn.LayerNorm((conv_out, seq_len)),
            nn.LeakyReLU(True),
        )
        self.cls_dim = sa_out + conv_out
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.6),
            nn.Linear(self.cls_dim, self.cls_dim // 4),
            nn.LeakyReLU(True),
            nn.Dropout(p=0.6),
            nn.Linear(self.cls_dim // 4, self.cls_dim // 4),
            nn.LeakyReLU(True),
            nn.Dropout(p=0.6),
            nn.Linear(self.cls_dim // 4, self.cls_dim // 64),
            nn.LeakyReLU(True),
            nn.Linear(self.cls_dim // 64, 1),
            nn.Sigmoid(),
        )
        self.regressor = nn.Sequential(
            nn.Dropout(p=0.6),
            nn.Linear(self.cls_dim, self.cls_dim // 4),
            nn.LeakyReLU(True),
            nn.Dropout(p=0.6),
            nn.Linear(self.cls_dim // 4, self.cls_dim // 4),
            nn.LeakyReLU(True),
            nn.Dropout(p=0.6),
            nn.Linear(self.cls_dim // 4, self.cls_dim // 64),
            nn.LeakyReLU(True),
            nn.Linear(self.cls_dim // 64, 1),
        )

    def forward(self, feats):
        out_sa = self.contextpooling(feats) + feats
        out_conv = self.conv(feats.permute(0, 2, 1)).permute(0, 2, 1) + feats
        out = torch.cat([out_sa, out_conv], dim=2)
        out = torch.max(out, dim=1)[0]
        return self.classifier(out).squeeze(-1), self.regressor(out).squeeze(-1)

class RBDSequenceDataset(Dataset):
    def __init__(self, seqs, seq_len=201):
        self.seqs = seqs
        self.max_len = seq_len + 2

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        seq = self.seqs[idx][: self.max_len - 2]
        tokens = [MAP_DICT['[CLS]']] + [MAP_DICT.get(ch, MAP_DICT['X']) for ch in seq] + [MAP_DICT['[SEP]']]
        mask = [1] * len(tokens)
        if len(tokens) < self.max_len:
            pad_len = self.max_len - len(tokens)
            tokens += [MAP_DICT['[PAD]']] * pad_len
            mask += [0] * pad_len
        return torch.tensor(tokens, dtype=torch.long), torch.tensor(mask, dtype=torch.float32)

## Load models and normalization statistics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

bert_model = CustomizeBertModel()
bert_model.load_from_ms_dict(torch.load(bert_ckpt_path, map_location='cpu'))
bert_model = bert_model.to(device).eval()

expr_model = SoluModel(seq_len=201, in_dim=1024).to(device).eval()
expr_model.load_state_dict(torch.load(expr_ckpt_path, map_location=device)['state_dict'])

bind_model = SoluModel(seq_len=201, in_dim=1024).to(device).eval()
bind_model.load_state_dict(torch.load(bind_ckpt_path, map_location=device)['state_dict'])

expr_mean = torch.from_numpy(np.load(os.path.join(stats_dir, 'expr/embedding_flatten_mean.npy'))).float().to(device)
expr_std = torch.from_numpy(np.load(os.path.join(stats_dir, 'expr/embedding_flatten_std.npy'))).float().to(device)
bind_mean = torch.from_numpy(np.load(os.path.join(stats_dir, 'bind/embedding_flatten_mean.npy'))).float().to(device)
bind_std = torch.from_numpy(np.load(os.path.join(stats_dir, 'bind/embedding_flatten_std.npy'))).float().to(device)

## Run prediction for each accession

os.makedirs(output_pred_dir, exist_ok=True)
txt_files = sorted(glob.glob(os.path.join(input_rbd_dir, '*.txt')))

for txt_path in txt_files:
    accession = os.path.splitext(os.path.basename(txt_path))[0]
    out_csv = os.path.join(output_pred_dir, f'pred_{accession}.csv')
    if os.path.exists(out_csv):
        print(f'[Skip] {out_csv} already exists.')
        continue

    with open(txt_path, 'r') as f:
        seqs = [line.strip() for line in f if line.strip()]

    dataset = RBDSequenceDataset(seqs, seq_len=201)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    expr_cls_all, expr_reg_all = [], []
    bind_cls_all, bind_reg_all = [], []

    with torch.no_grad():
        for input_ids, input_mask in tqdm(loader, desc=accession):
            input_ids = input_ids.to(device, non_blocking=True)
            input_mask = input_mask.to(device, non_blocking=True)

            with torch.amp.autocast('cuda', dtype=torch.float16):
                emb = bert_model(input_ids, input_mask)[:, 1:-1, :]
            emb_flat = emb.float().reshape(emb.shape[0], -1)

            expr_feats = ((emb_flat - expr_mean) / expr_std).view(-1, 201, 1024)
            e_cls, e_reg = expr_model(expr_feats)
            expr_cls_all.append(e_cls.cpu().numpy())
            expr_reg_all.append(e_reg.cpu().numpy())

            bind_feats = ((emb_flat - bind_mean) / bind_std).view(-1, 201, 1024)
            b_cls, b_reg = bind_model(bind_feats)
            bind_cls_all.append(b_cls.cpu().numpy())
            bind_reg_all.append(b_reg.cpu().numpy())

    df = pd.DataFrame({
        'sequence': seqs,
        'bind_cls': np.concatenate(bind_cls_all),
        'bind_reg': np.concatenate(bind_reg_all),
        'expr_cls': np.concatenate(expr_cls_all),
        'expr_reg': np.concatenate(expr_reg_all),
    })
    df.to_csv(out_csv, index=False)
    print(f'Saved predictions to {out_csv}')

## Ranking the predicted scores

In [ ]:
predicted_values_path = 'Path to the directory containing the predicted csvs'
validation_source_sequences = 'File containing the validation source sequences'
save_path = 'Path to save the top 20 candidates fasta files'

In [ ]:
import os 

def read_fasta(fasta_file):
    sequence_types = {}
    with open(fasta_file, 'r') as f:
        first = True
        lines = f.readlines()
        for line in lines:
            if line.startswith('>'):
                if not first:
                    sequence_types[line_category] = sequence
                else:
                    first = False
                line_category = line.strip()[1:]
                sequence = ''
            else:
                sequence += line.strip()
        sequence_types[line_category] = sequence
    return sequence_types

pairs = read_fasta(validation_source_sequences)

values = {}

for accession in list(pairs.keys()):
    values[accession] = []
    predictions_file = os.path.join(predicted_values_path, f'pred_{accession}.csv')
    with open(predictions_file, 'r') as f:
        _ = f.readline()
        for line in f.readlines():
            line_sp = line.strip().split(',')
            values[accession].append(line_sp)

top_10_binds = {}
top_10_exprs = {}

top_20_candidates_binds = {}
top_20_candidates_exprs = {}

for accession in list(values.keys()):
    sort_bind = sorted(values[accession], key=lambda x: float(x[1]), reverse=True)
    sort_expr = sorted(values[accession], key=lambda x: float(x[3]), reverse=True)

    top_10_bind = sort_bind[:int(.1*len(sort_bind))]
    top_10_expr = sort_expr[:int(.1*len(sort_expr))]

    top_10_binds[accession] = top_10_bind
    top_10_exprs[accession] = top_10_expr

    top_20_candidates_binds[accession] = sorted(top_10_bind, key=lambda x: float(x[2]), reverse=True)[:20]
    top_20_candidates_exprs[accession] = sorted(top_10_expr, key=lambda x: float(x[4]), reverse=True)[:20]

for accession in list(values.keys()):
    with open(os.path.join(save_path, f'{accession}/top_20_candidates_bind.fasta'), 'w') as f:
        for i,candidate in enumerate(top_20_candidates_binds[accession]):
            f.write(f'>{i}\n{candidate[0]}\n')

    with open(os.path.join(save_path, f'{accession}/top_20_candidates_expr.fasta'), 'w') as f:
        for i,candidate in enumerate(top_20_candidates_exprs[accession]):
            f.write(f'>{i}\n{candidate[0]}\n')